# Cardiovascular Disease Dataset — Preprocessing Notebook

## Objective
This notebook contains **only the EDA, data validation, cleaning, feature engineering, and preprocessing steps** for the cardiovascular dataset.

**No regression algorithms are implemented in this notebook.**

### Dataset
- 70,000 records
- 13 original features
- Regression target considered for the project: `ap_hi` (systolic blood pressure)
- `ap_hi` is excluded from the input features to prevent target leakage.

### Preprocessing workflow
1. Load the dataset
2. Inspect shape, data types and data quality
3. Check missing values and duplicates
4. Check valid ranges/domain constraints
5. Perform EDA and visualizations
6. Detect outliers using the IQR method
7. Clean invalid observations
8. Remove identifier column
9. Convert age from days to years
10. Engineer BMI
11. Validate categorical variables
12. Define features and target
13. Perform an 80:20 stratified split
14. Build train-only preprocessing pipelines
15. Transform training and test data
16. Validate the final preprocessed data


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42
DATA_PATH = "/mnt/data/05748e2d-c491-4698-8658-f91817eda947.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()


## 1. Basic Dataset Audit
We inspect the structure before making any changes.

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_count"))

print("\nExact duplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df["id"].duplicated().sum())

print("\nDescriptive statistics:")
display(df.describe().T)


## 2. Feature Description

In [ ]:
feature_description = pd.DataFrame({
    "Feature": [
        "id", "age", "gender", "height", "weight", "ap_hi", "ap_lo",
        "cholesterol", "gluc", "smoke", "alco", "active", "cardio"
    ],
    "Description": [
        "Unique patient identifier",
        "Age in days",
        "Gender category",
        "Height in centimeters",
        "Weight in kilograms",
        "Systolic blood pressure",
        "Diastolic blood pressure",
        "Cholesterol level category",
        "Glucose level category",
        "Smoking indicator",
        "Alcohol consumption indicator",
        "Physical activity indicator",
        "Cardiovascular disease indicator"
    ]
})
display(feature_description)


## 3. Distribution Visualizations

In [ ]:
df_eda = df.copy()
df_eda["age_years"] = df_eda["age"] / 365.25

numeric_features = ["age_years", "height", "weight", "ap_hi", "ap_lo"]

for col in numeric_features:
    plt.figure(figsize=(8, 5))
    sns.histplot(df_eda[col], bins=40, kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


In [ ]:
categorical_features = [
    "gender", "cholesterol", "gluc", "smoke", "alco", "active", "cardio"
]

for col in categorical_features:
    plt.figure(figsize=(7, 4))
    sns.countplot(data=df_eda, x=col)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


### Target Distribution: `ap_hi`

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df_eda["ap_hi"], bins=60, kde=True)
plt.title("Distribution of Systolic Blood Pressure (ap_hi)")
plt.xlabel("Systolic Blood Pressure (ap_hi)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 3.5))
sns.boxplot(x=df_eda["ap_hi"])
plt.title("Boxplot of Systolic Blood Pressure (ap_hi)")
plt.xlabel("ap_hi")
plt.tight_layout()
plt.show()


## 4. Correlation Analysis

In [ ]:
plt.figure(figsize=(11, 8))
corr = df_eda.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


## 5. Feature–Target Relationships

In [ ]:
scatter_features = ["age_years", "weight", "ap_lo"]

for col in scatter_features:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df_eda.sample(min(10000, len(df_eda)), random_state=RANDOM_STATE),
                    x=col, y="ap_hi", alpha=0.35)
    plt.title(f"{col} vs Systolic Blood Pressure")
    plt.xlabel(col)
    plt.ylabel("ap_hi")
    plt.tight_layout()
    plt.show()


In [ ]:
for col in categorical_features:
    if col != "cardio":
        plt.figure(figsize=(7, 5))
        sns.boxplot(data=df_eda, x=col, y="ap_hi")
        plt.title(f"{col} vs Systolic Blood Pressure")
        plt.xlabel(col)
        plt.ylabel("ap_hi")
        plt.tight_layout()
        plt.show()


## 6. Domain Validation

The following checks are used to identify clearly invalid or implausible records before modeling.

Rules used:
- Age: 18–100 years after conversion
- Height: 100–250 cm
- Weight: 30–200 kg
- Systolic BP (`ap_hi`): 70–250
- Diastolic BP (`ap_lo`): 40–150
- Systolic BP must be greater than diastolic BP
- `gender`: 1 or 2
- `cholesterol`: 1, 2 or 3
- `gluc`: 1, 2 or 3
- `smoke`, `alco`, `active`, `cardio`: 0 or 1


In [ ]:
domain_report = pd.DataFrame({
    "Rule": [
        "Age outside 18–100 years",
        "Height outside 100–250 cm",
        "Weight outside 30–200 kg",
        "ap_hi outside 70–250",
        "ap_lo outside 40–150",
        "ap_hi <= ap_lo"
    ],
    "Invalid_Count": [
        ((df_eda["age_years"] < 18) | (df_eda["age_years"] > 100)).sum(),
        ((df["height"] < 100) | (df["height"] > 250)).sum(),
        ((df["weight"] < 30) | (df["weight"] > 200)).sum(),
        ((df["ap_hi"] < 70) | (df["ap_hi"] > 250)).sum(),
        ((df["ap_lo"] < 40) | (df["ap_lo"] > 150)).sum(),
        (df["ap_hi"] <= df["ap_lo"]).sum()
    ]
})
display(domain_report)


In [ ]:
categorical_rules = {
    "gender": {1, 2},
    "cholesterol": {1, 2, 3},
    "gluc": {1, 2, 3},
    "smoke": {0, 1},
    "alco": {0, 1},
    "active": {0, 1},
    "cardio": {0, 1}
}

categorical_validation = []
for col, valid_values in categorical_rules.items():
    invalid_count = (~df[col].isin(valid_values)).sum()
    categorical_validation.append([col, sorted(valid_values), invalid_count])

categorical_validation = pd.DataFrame(
    categorical_validation,
    columns=["Feature", "Expected Values", "Invalid Count"]
)
display(categorical_validation)


## 7. IQR Outlier Detection
The IQR rule is used as a diagnostic to quantify statistical outliers. Domain-invalid values are handled using the domain rules above rather than blindly treating every IQR outlier as an error.

In [ ]:
iqr_features = ["age_years", "height", "weight", "ap_hi", "ap_lo"]

iqr_rows = []
for col in iqr_features:
    q1 = df_eda[col].quantile(0.25)
    q3 = df_eda[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df_eda[col] < lower) | (df_eda[col] > upper)).sum()
    iqr_rows.append([col, q1, q3, iqr, lower, upper, count, count / len(df_eda) * 100])

iqr_report = pd.DataFrame(
    iqr_rows,
    columns=["Feature", "Q1", "Q3", "IQR", "Lower Bound", "Upper Bound",
             "Outlier Count", "Outlier %"]
)
display(iqr_report)


In [ ]:
for col in iqr_features:
    plt.figure(figsize=(8, 3.5))
    sns.boxplot(x=df_eda[col])
    plt.title(f"Outlier Check — {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()


## 8. Data Cleaning

Cleaning decisions:
- Remove exact duplicate rows.
- Remove duplicate IDs if present.
- Drop `id` because it is an identifier and has no meaningful predictive interpretation.
- Convert age from days to years.
- Create BMI = weight / height².
- Keep only records satisfying the defined domain constraints.
- Do not clip the target using arbitrary statistical limits; invalid domain values are removed.


In [ ]:
clean_df = df.copy()

before_rows = len(clean_df)

# Remove exact duplicate rows
clean_df = clean_df.drop_duplicates()

# Remove duplicate IDs
clean_df = clean_df.drop_duplicates(subset="id", keep="first")

# Drop identifier
clean_df = clean_df.drop(columns=["id"])

# Convert age from days to years
clean_df["age_years"] = clean_df["age"] / 365.25
clean_df = clean_df.drop(columns=["age"])

# Domain validation
valid_mask = (
    clean_df["age_years"].between(18, 100)
    & clean_df["height"].between(100, 250)
    & clean_df["weight"].between(30, 200)
    & clean_df["ap_hi"].between(70, 250)
    & clean_df["ap_lo"].between(40, 150)
    & (clean_df["ap_hi"] > clean_df["ap_lo"])
    & clean_df["gender"].isin([1, 2])
    & clean_df["cholesterol"].isin([1, 2, 3])
    & clean_df["gluc"].isin([1, 2, 3])
    & clean_df["smoke"].isin([0, 1])
    & clean_df["alco"].isin([0, 1])
    & clean_df["active"].isin([0, 1])
    & clean_df["cardio"].isin([0, 1])
)

clean_df = clean_df.loc[valid_mask].copy()

# Feature engineering: BMI
clean_df["height_m"] = clean_df["height"] / 100
clean_df["BMI"] = clean_df["weight"] / (clean_df["height_m"] ** 2)
clean_df = clean_df.drop(columns=["height_m"])

print("Rows before cleaning:", before_rows)
print("Rows after cleaning:", len(clean_df))
print("Rows removed:", before_rows - len(clean_df))
print("Final shape after cleaning and feature engineering:", clean_df.shape)

display(clean_df.head())


## 9. Post-Cleaning Validation

In [ ]:
print("Missing values after cleaning:")
display(clean_df.isna().sum().to_frame("missing_count"))

print("Exact duplicate rows after cleaning:", clean_df.duplicated().sum())

print("\nCleaned descriptive statistics:")
display(clean_df.describe().T)


In [ ]:
post_clean_features = ["age_years", "height", "weight", "BMI", "ap_hi", "ap_lo"]

for col in post_clean_features:
    plt.figure(figsize=(8, 3.5))
    sns.boxplot(x=clean_df[col])
    plt.title(f"Post-Cleaning Boxplot — {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()


In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(clean_df.corr(numeric_only=True), annot=True, fmt=".2f",
            cmap="coolwarm", center=0)
plt.title("Correlation Heatmap After Cleaning")
plt.tight_layout()
plt.show()


## 10. Define Regression Features and Target

The selected regression target is:

**`ap_hi` — systolic blood pressure**

`ap_hi` must not appear in the feature matrix `X`.

The engineered `BMI` feature is included because it combines weight and height into a commonly interpretable body-size measure.

A pulse-pressure feature such as `ap_hi - ap_lo` is deliberately **not** created because it would contain the target `ap_hi` and cause target leakage.


In [ ]:
TARGET = "ap_hi"

X = clean_df.drop(columns=[TARGET]).copy()
y = clean_df[TARGET].copy()

assert TARGET not in X.columns, "Target leakage: ap_hi is present in X."

print("Target:", TARGET)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Input features:")
print(list(X.columns))


## 11. 80:20 Train/Test Split

The project guidelines require a stratified split. Because this is a regression target, temporary quantile bins of `ap_hi` are created **only for stratification**. The actual regression target remains continuous.

The split uses `random_state=42`.


In [ ]:
# Temporary bins used only to create approximately stratified regression splits
stratify_bins = pd.qcut(y, q=10, duplicates="drop")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=stratify_bins
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

print("\nTraining target summary:")
display(y_train.describe())

print("\nTesting target summary:")
display(y_test.describe())


## 12. Identify Numerical and Categorical Features

Preprocessing is separated by data type:
- Numerical features → missing-value handling + scaling where appropriate
- Categorical features → missing-value handling + one-hot encoding

The transformations are placed inside scikit-learn pipelines so that fitting occurs using the training data only.


In [ ]:
categorical_features = [
    "gender", "cholesterol", "gluc", "smoke", "alco", "active", "cardio"
]

numeric_features = [
    "height", "weight", "ap_lo", "age_years", "BMI"
]

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)


## 13. Train-Only Preprocessing Pipelines

In [ ]:
# Scaled preprocessing pipeline
numeric_pipeline_scaled = Pipeline([
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor_scaled = ColumnTransformer([
    ("num", numeric_pipeline_scaled, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Tree-based models do not require numerical scaling.
numeric_pipeline_unscaled = Pipeline([])

preprocessor_unscaled = ColumnTransformer([
    ("num", numeric_pipeline_unscaled, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessing pipelines created successfully.")
print("Important: these transformers are not fitted on the complete dataset.")


## 14. Fit Preprocessing on Training Data Only

The following transformation demonstrates the preprocessing stage without training any regression model.

- `fit_transform()` is used on `X_train`.
- `transform()` is used on `X_test`.
- The test set is therefore not used to estimate scaling or encoding parameters.


In [ ]:
X_train_processed = preprocessor_scaled.fit_transform(X_train)
X_test_processed = preprocessor_scaled.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

print("Processed data type:", type(X_train_processed))


In [ ]:
# Recover feature names after one-hot encoding
feature_names = preprocessor_scaled.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Number of final model-ready features:", len(feature_names))
display(X_train_processed_df.head())


## 15. Final Preprocessing Validation

In [ ]:
print("Final validation checks")

print("1. Missing values in cleaned data:",
      int(clean_df.isna().sum().sum()))

print("2. Duplicate rows in cleaned data:",
      int(clean_df.duplicated().sum()))

print("3. Target leakage:",
      TARGET in X.columns)

print("4. Processed train rows:", len(X_train_processed_df))
print("5. Processed test rows:", len(X_test_processed_df))

print("6. Missing values in processed training matrix:",
      int(np.isnan(X_train_processed).sum()))

print("7. Missing values in processed testing matrix:",
      int(np.isnan(X_test_processed).sum()))

assert TARGET not in X.columns
assert not np.isnan(X_train_processed).any()
assert not np.isnan(X_test_processed).any()

print("\nAll preprocessing validation checks passed.")


# Preprocessing Summary

The dataset has now passed through the complete preprocessing stage:

- Dataset structure audited
- Missing values checked
- Duplicate rows and IDs checked
- Domain/range validity checked
- Categorical values validated
- Statistical outliers identified using IQR
- Invalid observations removed according to defined domain rules
- `id` removed
- Age converted from days to years
- BMI engineered
- Target `ap_hi` separated from predictors
- Target leakage checked
- 80:20 stratified train/test split performed
- Numerical and categorical preprocessing pipelines created
- Scaling/encoding fitted on training data only
- Test data transformed without fitting
- Final processed matrices validated

**No regression algorithms are included in this notebook.**

The resulting `X_train_processed`, `X_test_processed`, `y_train`, and `y_test` are ready to be supplied to the regression-model notebook.
